# 03 -- Spectral Analysis Demo

Compute transformation matrices W_i for each dialect, eigendecompose them,
visualise eigenspectra, and compute pairwise spectral distances.

**Key modules:** `spectral.transformation`, `spectral.eigendecomposition`,
`spectral.eigenspectrum`, `spectral.entropy`, `spectral.distance`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from eigendialectos.constants import DialectCode, DIALECT_NAMES
from eigendialectos.types import EmbeddingMatrix
from eigendialectos.spectral.transformation import (
    compute_transformation_matrix, compute_all_transforms,
)
from eigendialectos.spectral.eigendecomposition import eigendecompose
from eigendialectos.spectral.eigenspectrum import compute_eigenspectrum
from eigendialectos.spectral.entropy import compute_dialectal_entropy, compare_entropies

## Setup: Synthetic Embeddings

In [ ]:
rng = np.random.default_rng(42)
dim, vocab_size = 30, 100
vocab = [f"w{i}" for i in range(vocab_size)]

base = rng.standard_normal((dim, vocab_size)).astype(np.float64)
embeddings = {}
for code in DialectCode:
    if code == DialectCode.ES_PEN:
        data = base.copy()
    else:
        scale = 0.1 + 0.2 * rng.random()
        data = base + rng.standard_normal((dim, vocab_size)) * scale
    embeddings[code] = EmbeddingMatrix(data=data, vocab=vocab, dialect_code=code)

print(f"Created {len(embeddings)} embedding spaces (dim={dim}, vocab={vocab_size})")

## Compute Transformation Matrices

W_i = E_i @ E_ref^T @ (E_ref @ E_ref^T + lambda*I)^{-1}

In [ ]:
transforms = compute_all_transforms(
    embeddings, reference=DialectCode.ES_PEN, method="lstsq", regularization=0.01
)

print(f"Computed {len(transforms)} transformation matrices")
for code, W in sorted(transforms.items(), key=lambda x: x[0].value):
    cond = np.linalg.cond(W.data)
    print(f"  {code.value}: shape={W.shape}, cond={cond:.1f}")

## Eigendecompose and Compute Spectra

In [ ]:
eigendecomps = {}
spectra = {}
entropies = {}

for code, W in transforms.items():
    eigen = eigendecompose(W)
    spectrum = compute_eigenspectrum(eigen)
    eigendecomps[code] = eigen
    spectra[code] = spectrum
    entropies[code] = spectrum.entropy
    print(
        f"{code.value}: rank={eigen.rank}, entropy={spectrum.entropy:.4f}, "
        f"top-5 eigenvalues={np.sort(np.abs(eigen.eigenvalues))[::-1][:5].real.round(3)}"
    )

## Eigenspectrum Visualisation

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
for idx, code in enumerate(sorted(spectra.keys(), key=lambda c: c.value)):
    ax = axes[idx // 4][idx % 4]
    ev = spectra[code].eigenvalues_sorted[:20]
    ax.bar(range(len(ev)), ev, color="steelblue")
    ax.set_title(f"{code.value}\nH={spectra[code].entropy:.3f}", fontsize=9)
    ax.set_xlabel("Index")
    if idx % 4 == 0:
        ax.set_ylabel("|eigenvalue|")

fig.suptitle("Eigenspectra of Dialect Transformation Matrices (top 20)", fontsize=12)
plt.tight_layout()
plt.show()

## Entropy Comparison

In [ ]:
comparison = compare_entropies(entropies)

fig, ax = plt.subplots(figsize=(10, 5))
codes_sorted = [c.value for c, _ in comparison["rankings"]]
vals_sorted = [v for _, v in comparison["rankings"]]
ax.barh(codes_sorted, vals_sorted, color="teal")
ax.set_xlabel("Spectral Entropy")
ax.set_title("Dialectal Spectral Entropy (descending)")
ax.axvline(comparison["mean"], color="red", linestyle="--", label=f"Mean={comparison['mean']:.3f}")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nInterpretation: {comparison['interpretation']}")

## Cumulative Energy Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for code in sorted(spectra.keys(), key=lambda c: c.value):
    ce = spectra[code].cumulative_energy
    ax.plot(range(len(ce)), ce, label=code.value, linewidth=1.5)

ax.axhline(0.9, color="gray", linestyle="--", alpha=0.5, label="90% energy")
ax.set_xlabel("Number of eigenvalues")
ax.set_ylabel("Cumulative energy")
ax.set_title("Cumulative Spectral Energy by Dialect")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## Pairwise Distance Matrix

In [ ]:
from eigendialectos.spectral.distance import frobenius_distance, compute_distance_matrix

D = compute_distance_matrix(
    transforms, spectra, entropies, method="frobenius"
)
labels = sorted(transforms.keys(), key=lambda c: c.value)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(D, cmap="YlOrRd")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([c.value for c in labels], rotation=45, ha="right")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels([c.value for c in labels])
fig.colorbar(im, ax=ax, label="Frobenius Distance")
ax.set_title("Pairwise Frobenius Distance Between Dialect Transforms")
plt.tight_layout()
plt.show()